# FoodSeq Reference Database Pipeline

This notebook builds two reference databases:
- **Part A — trnL**: Plant sequences amplified by the trnLg/trnLh primer pair, combining RefSeq plastid genomes and GenBank
- **Part B — 12SV5**: Vertebrate dietary sequences amplified by the 12SV5 primer pair, combining RefSeq mitochondrial genomes and GenBank

Both pipelines share the same setup, SLURM jobs, and taxonomy approach.

**Before running:** update the paths in the Configuration cell below to match your cluster environment.

---
## 0. Configuration

Update these paths and parameters before running anything else. All other cells reference these variables.

In [ ]:
# ── Cluster paths ─────────────────────────────────────────────────────────────
SCRATCH       <- "/scratch/your_username"                  # cluster scratch directory
REPO_DIR      <- "/path/to/food-dbs"                       # cloned repo root
PLASTID_DIR   <- file.path(SCRATCH, "plastid_refseq")      # RefSeq plastid download
MITO_DIR      <- file.path(SCRATCH, "mito_refseq")         # RefSeq mitochondrial download
SQL_PATH      <- file.path(SCRATCH, "accessionTaxa.sql")   # taxonomizr SQL database

# ── SLURM parameters ──────────────────────────────────────────────────────────
PARTITION     <- "general"    # partition/queue name; check with 'sinfo'
R_MODULE      <- "R/4.3.0"    # R module name; check with 'module avail R'

# ── Output directories ────────────────────────────────────────────────────────
TRNL_DADA2   <- file.path(REPO_DIR, "data", "outputs", "dada2-compatible", "trnL")
TRNL_QIIME2  <- file.path(REPO_DIR, "data", "outputs", "qiime2-compatible", "trnL")
S12V5_DADA2  <- file.path(REPO_DIR, "data", "outputs", "dada2-compatible", "12Sv5")
S12V5_QIIME2 <- file.path(REPO_DIR, "data", "outputs", "qiime2-compatible", "12Sv5")

for (d in c(TRNL_DADA2, TRNL_QIIME2, S12V5_DADA2, S12V5_QIIME2)) dir.create(d, recursive = TRUE, showWarnings = FALSE)

cat("Configuration set.\n")
cat("Repo:   ", REPO_DIR, "\n")
cat("Scratch:", SCRATCH,  "\n")

## 1. Install R packages (one-time setup)

Run once on first use; safe to skip on subsequent runs.

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install(c("Biostrings", "ShortRead"), ask = FALSE)
install.packages(c("taxonomizr", "tidyverse", "rentrez", "here"),
                 repos = "https://cloud.r-project.org")

cat("Package installation complete.\n")

---
## 2. Submit SLURM jobs

All three jobs can run simultaneously. Submit them all, then monitor progress in section 3 before proceeding to Part A or B.

### 2a. Download RefSeq plastid sequences (for trnL)

In [ ]:
plastid_script <- paste0(
"#!/bin/bash\n",
"#SBATCH --job-name=refseq_plastid\n",
"#SBATCH --partition=", PARTITION, "\n",
"#SBATCH --time=04:00:00\n",
"#SBATCH --mem=8G\n",
"#SBATCH --cpus-per-task=2\n",
"#SBATCH --output=", SCRATCH, "/refseq_plastid.log\n",
"\n",
"mkdir -p ", PLASTID_DIR, "\n",
"wget -r -nd -A '*.genomic.fna.gz' \\\n",
"  ftp://ftp.ncbi.nlm.nih.gov/refseq/release/plastid/ \\\n",
"  -P ", PLASTID_DIR, "\n",
"zcat ", PLASTID_DIR, "/*.genomic.fna.gz > ", PLASTID_DIR, "/refseq_plastid_all.fasta\n",
"echo 'Plastid download complete.'\n"
)

plastid_sh <- file.path(SCRATCH, "download_plastid.sh")
writeLines(plastid_script, plastid_sh)

result <- system(paste("sbatch", plastid_sh), intern = TRUE)
plastid_job_id <- gsub("[^0-9]", "", result)
cat(result, "\n")
cat("Plastid job ID:", plastid_job_id, "\n")

### 2b. Download RefSeq mitochondrial sequences (for 12SV5)

In [ ]:
mito_script <- paste0(
"#!/bin/bash\n",
"#SBATCH --job-name=refseq_mito\n",
"#SBATCH --partition=", PARTITION, "\n",
"#SBATCH --time=04:00:00\n",
"#SBATCH --mem=8G\n",
"#SBATCH --cpus-per-task=2\n",
"#SBATCH --output=", SCRATCH, "/refseq_mito.log\n",
"\n",
"mkdir -p ", MITO_DIR, "\n",
"wget -r -nd -A '*.genomic.fna.gz' \\\n",
"  ftp://ftp.ncbi.nlm.nih.gov/refseq/release/mitochondrion/ \\\n",
"  -P ", MITO_DIR, "\n",
"zcat ", MITO_DIR, "/*.genomic.fna.gz > ", MITO_DIR, "/refseq_mito_all.fasta\n",
"echo 'Mitochondrial download complete.'\n"
)

mito_sh <- file.path(SCRATCH, "download_mito.sh")
writeLines(mito_script, mito_sh)

result <- system(paste("sbatch", mito_sh), intern = TRUE)
mito_job_id <- gsub("[^0-9]", "", result)
cat(result, "\n")
cat("Mitochondrial job ID:", mito_job_id, "\n")

### 2c. Build taxonomizr SQL database (shared by both pipelines)

In [ ]:
taxonomizr_script <- paste0(
"#!/bin/bash\n",
"#SBATCH --job-name=taxonomizr_build\n",
"#SBATCH --partition=", PARTITION, "\n",
"#SBATCH --time=12:00:00\n",
"#SBATCH --mem=16G\n",
"#SBATCH --cpus-per-task=4\n",
"#SBATCH --output=", SCRATCH, "/taxonomizr_build.log\n",
"\n",
"module load ", R_MODULE, "\n",
"\n",
"Rscript -e \"\n",
"  library(taxonomizr)\n",
"  prepareDatabase('", SQL_PATH, "')\n",
"  cat('taxonomizr database built.\\n')\n",
"\""
)

tax_sh <- file.path(SCRATCH, "build_taxonomizr.sh")
writeLines(taxonomizr_script, tax_sh)

result <- system(paste("sbatch", tax_sh), intern = TRUE)
taxonomizr_job_id <- gsub("[^0-9]", "", result)
cat(result, "\n")
cat("taxonomizr job ID:", taxonomizr_job_id, "\n")

## 3. Monitor jobs

Re-run this cell to check status. Proceed to Part A or B once the relevant jobs are complete (no longer shown in the queue).

In [ ]:
status <- system("squeue --me --format='%.18i %.9P %.30j %.8T %.10M %.6D %R'", intern = TRUE)
cat(paste(status, collapse = "\n"), "\n")

logs <- list(
    plastid    = file.path(SCRATCH, "refseq_plastid.log"),
    mito       = file.path(SCRATCH, "refseq_mito.log"),
    taxonomizr = file.path(SCRATCH, "taxonomizr_build.log")
)

for (name in names(logs)) {
    if (file.exists(logs[[name]])) {
        cat("\n---", name, "(last 3 lines) ---\n")
        cat(paste(tail(readLines(logs[[name]]), 3), collapse = "\n"), "\n")
    }
}

## 4. Load shared libraries and functions

Run this before starting either Part A or Part B.

In [ ]:
library(Biostrings)
library(ShortRead)
library(taxonomizr)
library(tidyverse)
library(rentrez)

source(file.path(REPO_DIR, "code", "functions", "find_primer_pair.R"))
source(file.path(REPO_DIR, "code", "functions", "query_ncbi.R"))

# Shared taxonomy extraction helper
extract_taxonomy <- function(ids, sql_path) {
    vars <- c("superkingdom", "phylum", "class", "order",
              "family", "genus", "species", "subspecies", "varietas", "forma")
    taxonomy.raw <- getRawTaxonomy(ids, sql_path)
    taxonomy <- data.frame(matrix(ncol = length(vars), nrow = 0))
    colnames(taxonomy) <- vars
    for (i in seq_along(taxonomy.raw)) {
        row.i  <- taxonomy.raw[[i]] |> t() |> data.frame()
        shared <- intersect(vars, names(row.i))
        row.i  <- select(row.i, one_of(shared))
        taxonomy <- bind_rows(taxonomy, row.i)
    }
    taxonomy$taxid <- names(taxonomy.raw) |> trimws() |> as.integer()
    select(taxonomy, taxid, everything())
}

cat("Libraries and functions loaded.\n")

---
# Part A — trnL Reference Database

Primers: trnLg (`GGGCAATCCTGAGCCAA`) / trnLh (`CCATTGAGTCTCTGCACCTATC`)  
Source organisms: curated food plant list (`human-foods.csv`)

## A1. Load trnL inputs

In [ ]:
trnLG <- DNAString("GGGCAATCCTGAGCCAA")
trnLH <- DNAString("CCATTGAGTCTCTGCACCTATC")
trnL_primers <- list(trnLG, trnLH)

plants <-
    file.path(REPO_DIR, "data", "inputs", "human-foods.csv") |>
    read.csv(stringsAsFactors = FALSE) |>
    filter(category == "plant") |>
    pull(scientific_name)

edits <-
    file.path(REPO_DIR, "data", "inputs", "Manual renaming.csv") |>
    read_csv()

cat(length(plants), "food plant species\n")
head(plants)

## A2. Filter RefSeq plastid to one sequence per species

In [ ]:
plastid_species_path <- file.path(PLASTID_DIR, "refseq_plastid_species.fasta")

if (!file.exists(plastid_species_path)) {
    cat("Filtering RefSeq plastid to one sequence per species...\n")
    plastid <- readDNAStringSet(file.path(PLASTID_DIR, "refseq_plastid_all.fasta"))
    cat(length(plastid), "total sequences\n")

    species <- sub("^\\S+\\s+(\\S+\\s+\\S+).*", "\\1", names(plastid))
    plastid_species <- plastid[!duplicated(species)]
    names(plastid_species) <- species[!duplicated(species)]

    writeXStringSet(plastid_species, plastid_species_path)
    cat(length(plastid_species), "species retained, saved to", plastid_species_path, "\n")
} else {
    cat("Species FASTA already exists, loading directly.\n")
    plastid_species <- readDNAStringSet(plastid_species_path)
    cat(length(plastid_species), "species loaded\n")
}

## A3. RefSeq: in silico PCR and subset to food plants

In [ ]:
refseq.trnL <- find_primer_pair(plastid_species,
                                fwd = trnL_primers[[1]],
                                rev = trnL_primers[[2]])
cat(length(refseq.trnL), "sequences contain the trnL g/h primer pair\n")

plants.i    <- lapply(plants, grep, x = names(refseq.trnL)) |> unlist()
refseq.trnL <- refseq.trnL[plants.i]
cat(length(refseq.trnL), "food plant trnL sequences from RefSeq\n")

names(refseq.trnL) <- gsub(" .*$", "", names(refseq.trnL))
head(names(refseq.trnL))

## A4. NCBI GenBank: remote query

In [ ]:
cat("Querying NCBI for trnL sequences... this may take 10-30 minutes\n")
ncbi.trnL <- query_ncbi(marker = "trnL", organisms = plants)
cat(length(ncbi.trnL), "sequences returned from NCBI\n")

In [ ]:
ncbi.trnL <- find_primer_pair(ncbi.trnL,
                              fwd = trnL_primers[[1]],
                              rev = trnL_primers[[2]])
cat(length(ncbi.trnL), "sequences contain the trnL g/h primer pair\n")

n_before  <- length(ncbi.trnL)
ncbi.trnL <- ncbi.trnL[!grepl("UNVERIFIED", names(ncbi.trnL))]
cat("Removed", n_before - length(ncbi.trnL), "unverified sequences\n")

names(ncbi.trnL) <-
    names(ncbi.trnL) |>
    gsub(pattern = " .+$", replacement = "") |>
    gsub(pattern = "^>",   replacement = "")
head(names(ncbi.trnL))

## A5. Combine RefSeq and GenBank

In [ ]:
cat("RefSeq sequences: ",  length(refseq.trnL), "\n")
cat("GenBank sequences:",  length(ncbi.trnL),   "\n")
cat("Shared accessions: ", intersect(names(ncbi.trnL), names(refseq.trnL)) |> length(), "\n")

trnL.df <- bind_rows(
    data.frame(source = "GenBank", accession = names(ncbi.trnL),   seq = as.character(ncbi.trnL)),
    data.frame(source = "RefSeq",  accession = names(refseq.trnL), seq = as.character(refseq.trnL))
)
cat(nrow(trnL.df), "total sequences\n")

In [ ]:
# Fetch manual additions from curation file
additions <- filter(edits, type == "add")

if (nrow(additions) > 0) {
    cat("Fetching", nrow(additions), "manual addition(s) from NCBI...\n")
    seqs_raw <-
        entrez_fetch(db = "nucleotide", id = additions$accession, rettype = "fasta") |>
        strsplit("\n{2,}") |> unlist()

    accs      <- str_extract(seqs_raw, "[^>]\\S*")
    seqs_only <- seqs_raw |> sub("^[^\n]*\n", "", x = _) |> gsub("\n", "", x = _)

    manual_seqs        <- DNAStringSet(seqs_only)
    names(manual_seqs) <- accs
    manual_seqs        <- find_primer_pair(manual_seqs, fwd = trnL_primers[[1]], rev = trnL_primers[[2]])

    trnL.df <- bind_rows(
        data.frame(seq = as.character(manual_seqs), accession = names(manual_seqs)),
        trnL.df
    )
    cat("Manual additions appended. Total:", nrow(trnL.df), "\n")
}

## A6. Taxonomy lookup

In [ ]:
cat("Looking up taxonomy for", nrow(trnL.df), "accessions...\n")
trnL_ids <- accessionToTaxa(trnL.df$accession, SQL_PATH)
cat("Missing taxonomy IDs:", sum(is.na(trnL_ids)), "\n")

if (any(is.na(trnL_ids))) {
    cat("Accessions with missing IDs (assign manually in the next cell):\n")
    print(trnL.df[is.na(trnL_ids), c("source", "accession")])
}

In [ ]:
# Manually assign taxon IDs for accessions added to NCBI after the SQL database was built.
# Look up the taxon ID at https://www.ncbi.nlm.nih.gov/taxonomy and add lines below.
# Example:
#   trnL_ids[12] <- 3747   # OP764691.1 Fragaria x ananassa

cat("Remaining missing IDs:", sum(is.na(trnL_ids)), "\n")

In [ ]:
trnL_taxonomy <- extract_taxonomy(trnL_ids, SQL_PATH)
trnL.df       <- bind_cols(trnL.df, trnL_taxonomy)
cat("Taxonomy joined.\n")
head(trnL.df)

## A7. Manual curation edits

In [ ]:
# Omissions
omit    <- filter(edits, type == "omit")
trnL.df <- filter(trnL.df, !(accession %in% omit$accession & seq %in% omit$sequence))
cat("After omissions:", nrow(trnL.df), "sequences\n")

# Brassica oleracea subspecies renamings
trnL.df$varietas[trnL.df$accession == "AB213010.1"] <- "Brassica oleracea var. capitata"
trnL.df$varietas[trnL.df$accession == "AC183493.1"] <- "Brassica oleracea var. alboglabra"
trnL.df$varietas[trnL.df$accession == "LR031874.1"] <- "Brassica oleracea var. italica"
trnL.df$varietas[trnL.df$accession == "LR031875.1"] <- "Brassica oleracea var. italica"
trnL.df$varietas[trnL.df$accession == "LR031876.1"] <- "Brassica oleracea var. italica"

# Assign lowest available taxonomic rank as display name
trnL.df <-
    trnL.df |>
    MButils::lowest_level() |>
    rename(taxon = "name") |>
    select(source, accession, taxon, taxid, superkingdom:forma, seq)

cat("Curation edits applied.\n")

## A8. Quality control and orientation

In [ ]:
# Flag and report sequences with degenerate nucleotides
trnL.df$N <- grepl("[AGCT]*[^AGCT]+", trnL.df$seq)
cat(sum(trnL.df$N), "sequences with degenerate nucleotides\n")

with_n <- trnL.df$taxon[trnL.df$N] |> unique()
trnL.df |>
    filter(taxon %in% with_n) |>
    group_by(taxon, N) |> count() |> ungroup() |>
    group_by(taxon) |> summarize(has_clean_backup = any(!N)) |>
    arrange(has_clean_backup)

In [ ]:
trnL.df <- filter(trnL.df, !grepl("[AGCT]*[^AGCT]+", seq))
cat(nrow(trnL.df), "sequences after removing degenerate sequences\n")

# Standardise orientation: detect reversed sequences and replace with reverse complement
fwd_err <- floor(0.2 * length(trnLG))
rev_err <- floor(0.2 * length(trnLH))
ref     <- DNAStringSet(trnL.df$seq)
names(ref) <- paste(trnL.df$accession, trnL.df$taxon)

fwd_matches <-
    vmatchPattern(trnLG, ref, max.mismatch = fwd_err, fixed = TRUE) |>
    as.data.frame() |> filter(start <= 1) |> mutate(type = "forward") |> select(group, type)

rev_matches <-
    vmatchPattern(trnLH, ref, max.mismatch = rev_err, fixed = TRUE) |>
    as.data.frame() |> filter(start <= 1) |> mutate(type = "reverse") |> select(group, type)

trnL.df <-
    bind_rows(fwd_matches, rev_matches) |> arrange(group) |> bind_cols(trnL.df) |>
    mutate(seq = ifelse(type == "reverse",
                        DNAStringSet(seq) |> reverseComplement() |> as.character(),
                        seq)) |>
    select(-c(type, group))

cat("Orientation standardised.\n")

## A9. Deduplicate and save

In [ ]:
trnL.df <-
    trnL.df |>
    group_by(superkingdom, phylum, class, order, family,
             genus, species, subspecies, varietas, forma, taxon, seq) |>
    arrange(desc(source), accession) |>
    summarize(accession = first(accession), .groups = "drop") |>
    arrange(taxon, accession)

cat(nrow(trnL.df), "sequences after deduplication\n")
cat(n_distinct(trnL.df$taxon), "unique taxa\n")

In [ ]:
# DADA2: sequence file
trnL_out <- DNAStringSet(trnL.df$seq)
names(trnL_out) <- paste(trnL.df$accession, trnL.df$taxon)
writeXStringSet(trnL_out, file.path(TRNL_DADA2, "trnLGH.fasta"))
cat("Saved trnLGH.fasta\n")

# DADA2: taxonomy file
trnL.df_tax   <- trnL.df |> unite(col = "lineage", superkingdom:forma, sep = ";")
names(trnL_out) <- trnL.df_tax$lineage
writeXStringSet(trnL_out, file.path(TRNL_DADA2, "trnLGH_taxonomy.fasta"))
cat("Saved trnLGH_taxonomy.fasta\n")

# QIIME2
trnL.df_tax <- mutate(trnL.df_tax, acc = make.unique(accession, sep = "_"))
names(trnL_out) <- trnL.df_tax$acc
writeXStringSet(trnL_out, file.path(TRNL_QIIME2, "trnL-sequences.fasta"))
trnL.df_tax |>
    select(`Feature ID` = acc, Taxon = lineage) |>
    write_delim(file.path(TRNL_QIIME2, "trnL-taxonomy.tsv"), delim = "\t")
cat("Saved QIIME2 files\n")

cat("\n=== trnL complete:", nrow(trnL.df), "sequences,", n_distinct(trnL.df$taxon), "taxa ===\n")

---
# Part B — 12SV5 Reference Database

Primers: V5F (`TAGAACAGGCTCCTCTAG`) / V5R (`TTAGATACCCCACTATGC`)  
Source organisms: curated list of dietary vertebrates (hardcoded below; extend as needed)

> **Note:** The original `acc_to_dada()` function in the repo has incomplete code and a hardcoded legacy path. This notebook uses the same taxonomizr-based approach as the trnL pipeline instead, which is more robust and uses the same SQL database.

## B1. Load 12SV5 inputs

In [ ]:
V5F <- DNAString("TAGAACAGGCTCCTCTAG")
V5R <- DNAString("TTAGATACCCCACTATGC")
s12v5_primers <- list(V5F, V5R)

# Extend this list to include additional dietary animals
animals <- c(
    "Homo sapiens",
    "Bos taurus",
    "Gallus gallus",
    "Meleagris gallopavo",
    "Sus scrofa",
    "Bison bison",
    "Salmo salar",
    "Oreochromis niloticus",
    "Capra hircus",
    "Epinephelus morio",
    "Coryphaena hippurus",
    "Gadus morhua",
    "Paralichthys lethostigma",
    "Callinectes sapidus",
    "Oncorhynchus kisutch",
    "Apis mellifera"
)

cat(length(animals), "dietary animal species\n")
animals

## B2. Filter RefSeq mitochondrial to one sequence per species

In [ ]:
mito_species_path <- file.path(MITO_DIR, "refseq_mito_species.fasta")

if (!file.exists(mito_species_path)) {
    cat("Filtering RefSeq mitochondrial to one sequence per species...\n")
    mito <- readDNAStringSet(file.path(MITO_DIR, "refseq_mito_all.fasta"))
    cat(length(mito), "total sequences\n")

    # Strip 'mitochondrion' suffix from names for cleaner matching
    names(mito) <- names(mito) |>
        strsplit(" mitochondrion") |>
        lapply(`[[`, 1) |>
        unlist()

    species <- sub("^\\S+\\s+(\\S+\\s+\\S+).*", "\\1", names(mito))
    mito_species <- mito[!duplicated(species)]
    names(mito_species) <- species[!duplicated(species)]

    writeXStringSet(mito_species, mito_species_path)
    cat(length(mito_species), "species retained, saved to", mito_species_path, "\n")
} else {
    cat("Species FASTA already exists, loading directly.\n")
    mito_species <- readDNAStringSet(mito_species_path)
    cat(length(mito_species), "species loaded\n")
}

## B3. RefSeq: in silico PCR and subset to dietary animals

In [ ]:
refseq.12SV5 <- find_primer_pair(clean(mito_species),
                                 fwd = s12v5_primers[[1]],
                                 rev = s12v5_primers[[2]])
cat(length(refseq.12SV5), "sequences contain the 12SV5 primer pair\n")

animals.i     <- lapply(animals, grep, x = names(refseq.12SV5)) |> unlist()
refseq.12SV5  <- refseq.12SV5[animals.i]
cat(length(refseq.12SV5), "dietary animal 12SV5 sequences from RefSeq\n")

# Report any animals not found in RefSeq
found <- sub("\\S+\\s", "", names(refseq.12SV5))
missing <- animals[sapply(animals, function(x) length(grep(x, found))) == 0]
if (length(missing) > 0) {
    cat("Not found in RefSeq (will rely on GenBank):", paste(missing, collapse = ", "), "\n")
}

names(refseq.12SV5) <- gsub(" .*$", "", names(refseq.12SV5))
head(names(refseq.12SV5))

## B4. NCBI GenBank: remote query

In [ ]:
cat("Querying NCBI for 12S sequences... this may take several minutes\n")
ncbi.12SV5 <- query_ncbi(marker = "12S", organisms = animals)
cat(length(ncbi.12SV5), "sequences returned from NCBI\n")

In [ ]:
ncbi.12SV5 <- find_primer_pair(clean(ncbi.12SV5),
                               fwd = s12v5_primers[[1]],
                               rev = s12v5_primers[[2]])
cat(length(ncbi.12SV5), "sequences contain the 12SV5 primer pair\n")

n_before   <- length(ncbi.12SV5)
ncbi.12SV5 <- ncbi.12SV5[!grepl("UNVERIFIED", names(ncbi.12SV5))]
cat("Removed", n_before - length(ncbi.12SV5), "unverified sequences\n")

names(ncbi.12SV5) <-
    names(ncbi.12SV5) |>
    gsub(pattern = " .+$", replacement = "") |>
    gsub(pattern = "^>",   replacement = "")
head(names(ncbi.12SV5))

## B5. Combine RefSeq and GenBank

In [ ]:
cat("RefSeq sequences: ",  length(refseq.12SV5), "\n")
cat("GenBank sequences:",  length(ncbi.12SV5),   "\n")
cat("Shared accessions: ", intersect(names(ncbi.12SV5), names(refseq.12SV5)) |> length(), "\n")

s12v5.df <- bind_rows(
    data.frame(source = "GenBank", accession = names(ncbi.12SV5),   seq = as.character(ncbi.12SV5)),
    data.frame(source = "RefSeq",  accession = names(refseq.12SV5), seq = as.character(refseq.12SV5))
)
cat(nrow(s12v5.df), "total sequences\n")

## B6. Taxonomy lookup

In [ ]:
cat("Looking up taxonomy for", nrow(s12v5.df), "accessions...\n")
s12v5_ids <- accessionToTaxa(s12v5.df$accession, SQL_PATH)
cat("Missing taxonomy IDs:", sum(is.na(s12v5_ids)), "\n")

if (any(is.na(s12v5_ids))) {
    cat("Accessions with missing IDs (assign manually in the next cell):\n")
    print(s12v5.df[is.na(s12v5_ids), c("source", "accession")])
}

In [ ]:
# Manually assign taxon IDs for any accessions missing above.
# Look up the taxon ID at https://www.ncbi.nlm.nih.gov/taxonomy
# Example:
#   s12v5_ids[5] <- 9606   # Homo sapiens

cat("Remaining missing IDs:", sum(is.na(s12v5_ids)), "\n")

In [ ]:
s12v5_taxonomy <- extract_taxonomy(s12v5_ids, SQL_PATH)
s12v5.df       <- bind_cols(s12v5.df, s12v5_taxonomy)
cat("Taxonomy joined.\n")
head(s12v5.df)

## B7. Quality control and orientation

In [ ]:
# Degenerate nucleotides — note: clean() in find_primer_pair already removed most,
# but check for any that slipped through
s12v5.df$N <- grepl("[AGCT]*[^AGCT]+", s12v5.df$seq)
cat(sum(s12v5.df$N), "sequences with degenerate nucleotides\n")
s12v5.df <- filter(s12v5.df, !s12v5.df$N) |> select(-N)
cat(nrow(s12v5.df), "sequences remaining\n")

# Assign lowest available taxonomic rank
s12v5.df <-
    s12v5.df |>
    MButils::lowest_level() |>
    rename(taxon = "name") |>
    select(source, accession, taxon, taxid, superkingdom:forma, seq)

cat("QC complete.\n")

In [ ]:
# Standardise orientation
fwd_err_12 <- floor(0.2 * length(V5F))
rev_err_12 <- floor(0.2 * length(V5R))
ref_12     <- DNAStringSet(s12v5.df$seq)
names(ref_12) <- paste(s12v5.df$accession, s12v5.df$taxon)

fwd_matches_12 <-
    vmatchPattern(V5F, ref_12, max.mismatch = fwd_err_12, fixed = TRUE) |>
    as.data.frame() |> filter(start <= 1) |> mutate(type = "forward") |> select(group, type)

rev_matches_12 <-
    vmatchPattern(V5R, ref_12, max.mismatch = rev_err_12, fixed = TRUE) |>
    as.data.frame() |> filter(start <= 1) |> mutate(type = "reverse") |> select(group, type)

s12v5.df <-
    bind_rows(fwd_matches_12, rev_matches_12) |> arrange(group) |> bind_cols(s12v5.df) |>
    mutate(seq = ifelse(type == "reverse",
                        DNAStringSet(seq) |> reverseComplement() |> as.character(),
                        seq)) |>
    select(-c(type, group))

cat("Orientation standardised.\n")

## B8. Deduplicate and save

In [ ]:
s12v5.df <-
    s12v5.df |>
    group_by(superkingdom, phylum, class, order, family,
             genus, species, subspecies, varietas, forma, taxon, seq) |>
    arrange(desc(source), accession) |>
    summarize(accession = first(accession), .groups = "drop") |>
    arrange(taxon, accession)

cat(nrow(s12v5.df), "sequences after deduplication\n")
cat(n_distinct(s12v5.df$taxon), "unique taxa\n")

In [ ]:
# DADA2: sequence file
s12v5_out <- DNAStringSet(s12v5.df$seq)
names(s12v5_out) <- paste(s12v5.df$accession, s12v5.df$taxon)
writeXStringSet(s12v5_out, file.path(S12V5_DADA2, "12Sv5.fasta"))
cat("Saved 12Sv5.fasta\n")

# DADA2: taxonomy file
s12v5.df_tax    <- s12v5.df |> unite(col = "lineage", superkingdom:forma, sep = ";")
names(s12v5_out) <- s12v5.df_tax$lineage
writeXStringSet(s12v5_out, file.path(S12V5_DADA2, "12Sv5_taxonomy.fasta"))
cat("Saved 12Sv5_taxonomy.fasta\n")

# QIIME2
s12v5.df_tax <- mutate(s12v5.df_tax, acc = make.unique(accession, sep = "_"))
names(s12v5_out) <- s12v5.df_tax$acc
writeXStringSet(s12v5_out, file.path(S12V5_QIIME2, "12Sv5-sequences.fasta"))
s12v5.df_tax |>
    select(`Feature ID` = acc, Taxon = lineage) |>
    write_delim(file.path(S12V5_QIIME2, "12Sv5-taxonomy.tsv"), delim = "\t")
cat("Saved QIIME2 files\n")

cat("\n=== 12SV5 complete:", nrow(s12v5.df), "sequences,", n_distinct(s12v5.df$taxon), "taxa ===\n")

---
## Final summary

In [ ]:
cat("========================================\n")
cat("FoodSeq Reference Database Build Summary\n")
cat("========================================\n\n")

cat("trnL (plant)\n")
cat("  Sequences:", nrow(trnL.df), "\n")
cat("  Taxa:     ", n_distinct(trnL.df$taxon), "\n")
cat("  Outputs:\n")
cat("   ", file.path(TRNL_DADA2,  "trnLGH.fasta"), "\n")
cat("   ", file.path(TRNL_DADA2,  "trnLGH_taxonomy.fasta"), "\n")
cat("   ", file.path(TRNL_QIIME2, "trnL-sequences.fasta"), "\n")
cat("   ", file.path(TRNL_QIIME2, "trnL-taxonomy.tsv"), "\n\n")

cat("12SV5 (vertebrate)\n")
cat("  Sequences:", nrow(s12v5.df), "\n")
cat("  Taxa:     ", n_distinct(s12v5.df$taxon), "\n")
cat("  Outputs:\n")
cat("   ", file.path(S12V5_DADA2,  "12Sv5.fasta"), "\n")
cat("   ", file.path(S12V5_DADA2,  "12Sv5_taxonomy.fasta"), "\n")
cat("   ", file.path(S12V5_QIIME2, "12Sv5-sequences.fasta"), "\n")
cat("   ", file.path(S12V5_QIIME2, "12Sv5-taxonomy.tsv"), "\n")